## Importing libraries


In [ ]:
import os

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

import os
import shutil

import pandas as pd

import numpy as np

from sklearn.preprocessing import StandardScaler

import torch

import torch.nn as nn

from google.colab import files


from google.colab import drive

SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

WORK_START = 8
WORK_END = 18

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15

STREAM_THRESHOLD_PERCENTILE = 0.95
ENSEMBLE_THRESHOLD_PERCENTILE = 0.95

OCEAN_LOW_PERCENTILE = 0.05
OCEAN_HIGH_PERCENTILE = 0.95

EPOCHS = 50
PATIENCE = 7
LEARNING_RATE = 0.001
BATCH_SIZE = 2048

DATE_FORMAT = "%m/%d/%Y %H:%M:%S"

print("Training device:", DEVICE)

Training device: cpu


## Mount Drive and define file paths

In [ ]:
DATA_FOLDER = "/content"

# Model outputs will be saved in this runtime folder.
OUTPUT_FOLDER = "/content/CERT_r6_2_outputs"

# Create the output folder if it does not already exist.
os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

# Create the full path to the logon dataset.
LOGON_PATH = os.path.join(
    DATA_FOLDER,
    "logon.csv"
)

# Create the full path to the device dataset.
DEVICE_PATH = os.path.join(
    DATA_FOLDER,
    "device.csv"
)

# Create the full path to the file-activity dataset.
FILE_PATH = os.path.join(
    DATA_FOLDER,
    "file.csv"
)

# Create the full path to the psychometric dataset.
PSYCHOMETRIC_PATH = os.path.join(
    DATA_FOLDER,
    "psychometric.csv"
)

# Store the expected files and paths together.
required_files = {
    "logon.csv": LOGON_PATH,
    "device.csv": DEVICE_PATH,
    "file.csv": FILE_PATH,
    "psychometric.csv": PSYCHOMETRIC_PATH
}

# Store the names of any files that cannot be found.
missing_files = []

# Check each expected path.
for filename, path in required_files.items():

    # Add the filename to the missing list when it is absent.
    if not os.path.exists(path):
        missing_files.append(filename)

# Stop the notebook when one or more files are missing.
if missing_files:
    raise FileNotFoundError(
        "Upload these files to the Colab session before continuing: "
        + ", ".join(missing_files)
    )

# Display the uploaded file sizes.
for filename, path in required_files.items():

    # Convert file size from bytes to megabytes.
    size_in_mb = os.path.getsize(path) / (1024 ** 2)

    print(
        filename,
        "-",
        round(size_in_mb, 2),
        "MB"
    )

print("\nAll required CERT files were found.")

logon.csv - 10.0 MB
device.csv - 18.0 MB
file.csv - 17.0 MB
psychometric.csv - 0.17 MB

All required CERT files were found.


## Load Selected CSV Columns

In [ ]:
def load_columns(path, required_columns):
    header = pd.read_csv(path, nrows=0)

    column_lookup = {
        column.strip().lower(): column
        for column in header.columns
    }

    missing_columns = [
        column
        for column in required_columns
        if column not in column_lookup
    ]

    if missing_columns:
        raise ValueError(
            f"{os.path.basename(path)} is missing: {missing_columns}. "
            f"Available columns: {list(header.columns)}"
        )

    original_column_names = [
        column_lookup[column]
        for column in required_columns
    ]

    dataframe = pd.read_csv(
        path,
        usecols=original_column_names,
        low_memory=False
    )

    dataframe.columns = [
        column.strip().lower()
        for column in dataframe.columns
    ]

    return dataframe

## Prepare timestamps and common event fields

In [ ]:
def prepare_events(dataframe):
    dataframe = dataframe.copy()

    parsed_dates = pd.to_datetime(
        dataframe["date"],
        format=DATE_FORMAT,
        errors="coerce"
    )

    if parsed_dates.isna().mean() > 0.01:
        parsed_dates = pd.to_datetime(
            dataframe["date"],
            errors="coerce"
        )

    dataframe["date"] = parsed_dates

    dataframe = dataframe.dropna(
        subset=["date", "user", "pc"]
    ).copy()

    dataframe["user"] = (
        dataframe["user"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    dataframe["pc"] = (
        dataframe["pc"]
        .astype(str)
        .str.strip()
    )

    dataframe["day"] = dataframe["date"].dt.normalize()
    dataframe["hour"] = dataframe["date"].dt.hour
    dataframe["weekday"] = dataframe["date"].dt.weekday

    dataframe["weekend"] = (
        dataframe["weekday"] >= 5
    ).astype(int)

    dataframe["after_hours"] = (
        (dataframe["hour"] < WORK_START)
        | (dataframe["hour"] >= WORK_END)
    ).astype(int)

    return dataframe

## Create logon features

In [ ]:
def build_logon_features(dataframe):
    dataframe = prepare_events(dataframe)

    dataframe["activity"] = (
        dataframe["activity"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    dataframe["is_logon"] = (
        dataframe["activity"] == "logon"
    ).astype(int)

    dataframe["is_logoff"] = (
        dataframe["activity"] == "logoff"
    ).astype(int)

    daily = dataframe.groupby(
        ["user", "day"],
        as_index=False
    ).agg(
        logon_total=("activity", "size"),
        logon_count=("is_logon", "sum"),
        logoff_count=("is_logoff", "sum"),
        logon_unique_pcs=("pc", "nunique"),
        logon_first_hour=("hour", "min"),
        logon_last_hour=("hour", "max"),
        logon_after_hours=("after_hours", "sum"),
        logon_weekend=("weekend", "max"),
        logon_hour_std=("hour", "std")
    )

    daily["logon_hour_std"] = (
        daily["logon_hour_std"].fillna(0)
    )

    daily["logon_after_hours_ratio"] = (
        daily["logon_after_hours"]
        / daily["logon_total"]
    )

    daily["logon_logoff_ratio"] = (
        daily["logon_count"]
        / (daily["logoff_count"] + 1)
    )

    return daily

## Creating device features

In [ ]:
def build_device_features(dataframe):
    dataframe = prepare_events(dataframe)

    dataframe["activity"] = (
        dataframe["activity"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    dataframe["is_connect"] = (
        dataframe["activity"] == "connect"
    ).astype(int)

    dataframe["is_disconnect"] = (
        dataframe["activity"] == "disconnect"
    ).astype(int)

    daily = dataframe.groupby(
        ["user", "day"],
        as_index=False
    ).agg(
        device_total=("activity", "size"),
        connect_count=("is_connect", "sum"),
        disconnect_count=("is_disconnect", "sum"),
        device_unique_pcs=("pc", "nunique"),
        device_first_hour=("hour", "min"),
        device_last_hour=("hour", "max"),
        device_after_hours=("after_hours", "sum"),
        device_weekend=("weekend", "max"),
        device_hour_std=("hour", "std")
    )

    daily["device_hour_std"] = (
        daily["device_hour_std"].fillna(0)
    )

    daily["device_after_hours_ratio"] = (
        daily["device_after_hours"]
        / daily["device_total"]
    )

    daily["connect_disconnect_ratio"] = (
        daily["connect_count"]
        / (daily["disconnect_count"] + 1)
    )

    return daily

## Creating file features

In [ ]:
def build_file_features(dataframe):
    dataframe = prepare_events(dataframe)

    dataframe["filename"] = (
        dataframe["filename"]
        .astype(str)
        .str.strip()
    )

    daily = dataframe.groupby(
        ["user", "day"],
        as_index=False
    ).agg(
        file_total=("filename", "size"),
        unique_files=("filename", "nunique"),
        file_unique_pcs=("pc", "nunique"),
        file_first_hour=("hour", "min"),
        file_last_hour=("hour", "max"),
        file_after_hours=("after_hours", "sum"),
        file_weekend=("weekend", "max"),
        file_hour_std=("hour", "std")
    )

    daily["file_hour_std"] = (
        daily["file_hour_std"].fillna(0)
    )

    daily["file_after_hours_ratio"] = (
        daily["file_after_hours"]
        / daily["file_total"]
    )

    return daily

## Loading and aggregrating the three datasets

In [ ]:
logon_raw = load_columns(
    LOGON_PATH,
    ["date", "user", "pc", "activity"]
)

logon_features = build_logon_features(logon_raw)
del logon_raw

device_raw = load_columns(
    DEVICE_PATH,
    ["date", "user", "pc", "activity"]
)

device_features = build_device_features(device_raw)
del device_raw

file_raw = load_columns(
    FILE_PATH,
    ["date", "user", "pc", "filename"]
)

file_features = build_file_features(file_raw)
del file_raw

print("Logon profiles:", len(logon_features))
print("Device profiles:", len(device_features))
print("File profiles:", len(file_features))

Logon profiles: 327353
Device profiles: 92393
File profiles: 15492


## Defining model features and chronological dates

In [ ]:
logon_features["day"] = pd.to_datetime(logon_features["day"])
device_features["day"] = pd.to_datetime(device_features["day"])
file_features["day"] = pd.to_datetime(file_features["day"])


print(
    "Logon date range:",
    logon_features["day"].min(),
    "to",
    logon_features["day"].max()
)

print(
    "Device date range:",
    device_features["day"].min(),
    "to",
    device_features["day"].max()
)

print(
    "File date range:",
    file_features["day"].min(),
    "to",
    file_features["day"].max()
)


COMMON_START_DAY = max(
    logon_features["day"].min(),
    device_features["day"].min(),
    file_features["day"].min()
)


COMMON_END_DAY = min(
    logon_features["day"].max(),
    device_features["day"].max(),
    file_features["day"].max()
)


if COMMON_START_DAY >= COMMON_END_DAY:
    raise ValueError(
        "The logon, device and file datasets do not have "
        "an overlapping date range. Check that all files "
        "come from the same CERT release."
    )


print(
    "\nCommon date range:",
    COMMON_START_DAY,
    "to",
    COMMON_END_DAY
)


logon_features = logon_features[
    (logon_features["day"] >= COMMON_START_DAY)
    & (logon_features["day"] <= COMMON_END_DAY)
].copy()


device_features = device_features[
    (device_features["day"] >= COMMON_START_DAY)
    & (device_features["day"] <= COMMON_END_DAY)
].copy()


file_features = file_features[
    (file_features["day"] >= COMMON_START_DAY)
    & (file_features["day"] <= COMMON_END_DAY)
].copy()


common_days = pd.date_range(
    start=COMMON_START_DAY,
    end=COMMON_END_DAY,
    freq="D"
)


if len(common_days) < 20:
    raise ValueError(
        "There are too few shared dates for a reliable "
        "training-validation-test split."
    )


train_end_position = int(
    len(common_days) * TRAIN_RATIO
)


validation_end_position = int(
    len(common_days)
    * (TRAIN_RATIO + VALIDATION_RATIO)
)


train_end_position = max(
    1,
    min(train_end_position, len(common_days) - 2)
)

validation_end_position = max(
    train_end_position + 1,
    min(validation_end_position, len(common_days) - 1)
)


TRAIN_END_DAY = common_days[train_end_position]


VALIDATION_END_DAY = common_days[
    validation_end_position
]


print("\nTraining period:")
print(COMMON_START_DAY, "to", TRAIN_END_DAY - pd.Timedelta(days=1))

print("\nValidation period:")
print(
    TRAIN_END_DAY,
    "to",
    VALIDATION_END_DAY - pd.Timedelta(days=1)
)

print("\nTest period:")
print(VALIDATION_END_DAY, "to", COMMON_END_DAY)

Logon date range: 2010-01-02 00:00:00 to 2010-01-26 00:00:00
Device date range: 2010-01-02 00:00:00 to 2010-01-26 00:00:00
File date range: 2010-01-02 00:00:00 to 2010-01-26 00:00:00

Common date range: 2010-01-02 00:00:00 to 2010-01-26 00:00:00

Training period:
2010-01-02 00:00:00 to 2010-01-18 00:00:00

Validation period:
2010-01-19 00:00:00 to 2010-01-22 00:00:00

Test period:
2010-01-23 00:00:00 to 2010-01-26 00:00:00


## split check

In [ ]:
def check_split_sizes(dataframe, dataset_name):
    train_rows = dataframe[
        dataframe["day"] < TRAIN_END_DAY
    ]

    validation_rows = dataframe[
        (dataframe["day"] >= TRAIN_END_DAY)
        & (dataframe["day"] < VALIDATION_END_DAY)
    ]

    test_rows = dataframe[
        dataframe["day"] >= VALIDATION_END_DAY
    ]

    print(f"\n{dataset_name}")
    print("Training rows:", len(train_rows))
    print("Validation rows:", len(validation_rows))
    print("Test rows:", len(test_rows))

    if train_rows.empty:
        raise ValueError(
            f"{dataset_name} training split is empty."
        )

    if validation_rows.empty:
        raise ValueError(
            f"{dataset_name} validation split is empty."
        )

    if test_rows.empty:
        raise ValueError(
            f"{dataset_name} test split is empty."
        )


check_split_sizes(
    logon_features,
    "Logon dataset"
)

check_split_sizes(
    device_features,
    "Device dataset"
)

check_split_sizes(
    file_features,
    "File dataset"
)


Logon dataset
Training rows: 45198
Validation rows: 16000
Test rows: 8466

Device dataset
Training rows: 6586
Validation rows: 2231
Test rows: 1248

File dataset
Training rows: 10558
Validation rows: 3635
Test rows: 1299


## Building the autoencoder

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_size)
        )

    def forward(self, values):
        compressed = self.encoder(values)
        reconstructed = self.decoder(compressed)

        return reconstructed

## Calculating the reconstruction error

In [ ]:
def reconstruction_errors(model, values):
    errors = []

    model.eval()

    with torch.no_grad():
        for start in range(0, len(values), BATCH_SIZE):
            batch = values[
                start:start + BATCH_SIZE
            ].to(DEVICE)

            reconstructed = model(batch)

            batch_errors = (
                (batch - reconstructed) ** 2
            ).mean(dim=1)

            errors.extend(
                batch_errors.cpu().numpy()
            )

    return np.array(errors)

## Converting raw scores into comparable percentiles

In [ ]:
def percentile_scores(reference_errors, new_errors):
    sorted_reference = np.sort(reference_errors)

    scores = np.searchsorted(
        sorted_reference,
        new_errors,
        side="right"
    ) / len(sorted_reference)

    return scores

## Training one autoencoder correctly

In [ ]:
def train_stream(dataframe, feature_columns, stream_name):
    dataframe = dataframe.sort_values(
        ["day", "user"]
    ).reset_index(drop=True)

    train_data = dataframe[
        dataframe["day"] < TRAIN_END_DAY
    ].copy()

    validation_data = dataframe[
        (dataframe["day"] >= TRAIN_END_DAY)
        & (dataframe["day"] < VALIDATION_END_DAY)
    ].copy()

    test_data = dataframe[
        dataframe["day"] >= VALIDATION_END_DAY
    ].copy()

    print(f"\nTraining {stream_name} autoencoder")
    print("Training rows:", len(train_data))
    print("Validation rows:", len(validation_data))
    print("Test rows:", len(test_data))

    if train_data.empty:
        raise ValueError(
            f"{stream_name} has no training rows. "
            "Check its date range."
        )

    if validation_data.empty:
        raise ValueError(
            f"{stream_name} has no validation rows. "
            "Check TRAIN_END_DAY and VALIDATION_END_DAY."
        )

    if test_data.empty:
        raise ValueError(
            f"{stream_name} has no test rows. "
            "Check the dataset's final date."
        )

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_data[feature_columns]
    )

    validation_scaled = scaler.transform(
        validation_data[feature_columns]
    )

    test_scaled = scaler.transform(
        test_data[feature_columns]
    )

    train_values = torch.tensor(
        train_scaled,
        dtype=torch.float32
    )

    validation_values = torch.tensor(
        validation_scaled,
        dtype=torch.float32
    )

    test_values = torch.tensor(
        test_scaled,
        dtype=torch.float32
    )

    torch.manual_seed(SEED)

    model = Autoencoder(
        len(feature_columns)
    ).to(DEVICE)

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    best_validation_loss = np.inf
    best_weights = None
    waiting_epochs = 0

    for epoch in range(EPOCHS):
        model.train()

        row_order = torch.randperm(
            len(train_values)
        )

        total_training_loss = 0

        for start in range(
            0,
            len(train_values),
            BATCH_SIZE
        ):
            row_numbers = row_order[
                start:start + BATCH_SIZE
            ]

            batch = train_values[
                row_numbers
            ].to(DEVICE)

            optimizer.zero_grad()

            reconstructed = model(batch)

            loss = loss_function(
                reconstructed,
                batch
            )

            loss.backward()
            optimizer.step()

            total_training_loss += (
                loss.item() * len(batch)
            )

        average_training_loss = (
            total_training_loss
            / len(train_values)
        )

        current_validation_errors = (
            reconstruction_errors(
                model,
                validation_values
            )
        )

        current_validation_loss = (
            current_validation_errors.mean()
        )

        if current_validation_loss < best_validation_loss:
            best_validation_loss = current_validation_loss

            best_weights = {
                name: value.detach().cpu().clone()
                for name, value
                in model.state_dict().items()
            }

            waiting_epochs = 0

        else:
            waiting_epochs += 1

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(
                stream_name,
                "epoch",
                epoch + 1,
                "training loss",
                round(average_training_loss, 6),
                "validation loss",
                round(current_validation_loss, 6)
            )

        if waiting_epochs >= PATIENCE:
            print(stream_name, "stopped at epoch", epoch + 1)
            break

    model.load_state_dict(best_weights)

    validation_errors = reconstruction_errors(
        model,
        validation_values
    )

    test_errors = reconstruction_errors(
        model,
        test_values
    )

    threshold = np.quantile(
        validation_errors,
        STREAM_THRESHOLD_PERCENTILE
    )

    validation_results = validation_data[
        ["user", "day"]
    ].reset_index(drop=True)

    validation_results[
        f"{stream_name}_score"
    ] = validation_errors

    validation_results[
        f"{stream_name}_percentile"
    ] = percentile_scores(
        validation_errors,
        validation_errors
    )

    validation_results[
        f"{stream_name}_flag"
    ] = (
        validation_errors > threshold
    ).astype(int)

    test_results = test_data[
        ["user", "day"]
    ].reset_index(drop=True)

    test_results[
        f"{stream_name}_score"
    ] = test_errors

    test_results[
        f"{stream_name}_percentile"
    ] = percentile_scores(
        validation_errors,
        test_errors
    )

    test_results[
        f"{stream_name}_flag"
    ] = (
        test_errors > threshold
    ).astype(int)

    return {
        "model": model,
        "scaler": scaler,
        "threshold": float(threshold),
        "validation": validation_results,
        "test": test_results
    }

## Training the three independent models

In [ ]:
logon_output = train_stream(
    logon_features,
    LOGON_FEATURES,
    "logon"
)

device_output = train_stream(
    device_features,
    DEVICE_FEATURES,
    "device"
)

file_output = train_stream(
    file_features,
    FILE_FEATURES,
    "file"
)


Training logon autoencoder
Training rows: 45198
Validation rows: 16000
Test rows: 8466
logon epoch 1 training loss 0.99646 validation loss 0.892787
logon epoch 10 training loss 0.139604 validation loss 0.114746
logon epoch 20 training loss 0.029422 validation loss 0.020174
logon epoch 30 training loss 0.015815 validation loss 0.012339
logon epoch 40 training loss 0.008777 validation loss 0.007083
logon epoch 50 training loss 0.005712 validation loss 0.004527

Training device autoencoder
Training rows: 6586
Validation rows: 2231
Test rows: 1248
device epoch 1 training loss 1.005747 validation loss 0.919096
device epoch 10 training loss 0.915658 validation loss 0.818589
device epoch 20 training loss 0.599761 validation loss 0.503494
device epoch 30 training loss 0.376375 validation loss 0.282103
device epoch 40 training loss 0.23731 validation loss 0.15382
device epoch 50 training loss 0.193256 validation loss 0.122412

Training file autoencoder
Training rows: 10558
Validation rows: 363

## Merging hte model scores and Creating the ensemble

In [ ]:
validation_ensemble = (
    logon_output["validation"]
    .merge(
        device_output["validation"],
        on=["user", "day"],
        how="outer"
    )
    .merge(
        file_output["validation"],
        on=["user", "day"],
        how="outer"
    )
)

test_ensemble = (
    logon_output["test"]
    .merge(
        device_output["test"],
        on=["user", "day"],
        how="outer"
    )
    .merge(
        file_output["test"],
        on=["user", "day"],
        how="outer"
    )
)

percentile_columns = [
    "logon_percentile",
    "device_percentile",
    "file_percentile"
]

flag_columns = [
    "logon_flag",
    "device_flag",
    "file_flag"
]

validation_ensemble["ensemble_score"] = (
    validation_ensemble[
        percentile_columns
    ].mean(axis=1, skipna=True)
)

test_ensemble["ensemble_score"] = (
    test_ensemble[
        percentile_columns
    ].mean(axis=1, skipna=True)
)

ensemble_threshold = np.quantile(
    validation_ensemble[
        "ensemble_score"
    ].dropna(),
    ENSEMBLE_THRESHOLD_PERCENTILE
)

for column in flag_columns:
    test_ensemble[column] = (
        test_ensemble[column]
        .fillna(0)
        .astype(int)
    )

test_ensemble["ensemble_flag"] = (
    test_ensemble["ensemble_score"]
    > ensemble_threshold
).astype(int)

test_ensemble["flagged_streams"] = (
    test_ensemble[flag_columns]
    .sum(axis=1)
)

print("Ensemble threshold:", ensemble_threshold)
print("Detected test anomalies:", test_ensemble["ensemble_flag"].sum())

Ensemble threshold: 0.9270062499999999
Detected test anomalies: 697


## Creating the technical anomaly categories

In [ ]:
def technical_category(row):
    if row["ensemble_flag"] == 0:
        return "Normal"

    if row["flagged_streams"] >= 2:
        return "Multi-Stream Anomaly"

    available_scores = {}

    for stream in ["logon", "device", "file"]:
        score = row[f"{stream}_percentile"]

        if not pd.isna(score):
            available_scores[stream] = score

    if not available_scores:
        return "Unclassified Technical Anomaly"

    dominant_stream = max(
        available_scores,
        key=available_scores.get
    )

    return dominant_stream.capitalize() + "-Dominant Anomaly"


test_ensemble["technical_category"] = (
    test_ensemble.apply(
        technical_category,
        axis=1
    )
)

## Loading and preparing the psychometric file

In [ ]:
psychometric = pd.read_csv(
    PSYCHOMETRIC_PATH,
    low_memory=False
)

psychometric.columns = [
    column.strip().lower()
    for column in psychometric.columns
]

if "user_id" in psychometric.columns:
    psychometric = psychometric.rename(
        columns={"user_id": "user"}
    )

elif "userid" in psychometric.columns:
    psychometric = psychometric.rename(
        columns={"userid": "user"}
    )

elif "user" not in psychometric.columns:
    raise ValueError(
        "No user ID column was found in psychometric.csv."
    )

required_ocean_columns = [
    "o",
    "c",
    "e",
    "a",
    "n"
]

missing_ocean_columns = [
    column
    for column in required_ocean_columns
    if column not in psychometric.columns
]

if missing_ocean_columns:
    raise ValueError(
        f"Missing OCEAN columns: {missing_ocean_columns}"
    )

psychometric = psychometric.rename(
    columns={
        "o": "openness",
        "c": "conscientiousness",
        "e": "extraversion",
        "a": "agreeableness",
        "n": "neuroticism"
    }
)

psychometric["user"] = (
    psychometric["user"]
    .astype(str)
    .str.strip()
    .str.upper()
)

OCEAN_TRAITS = [
    "openness",
    "conscientiousness",
    "extraversion",
    "agreeableness",
    "neuroticism"
]

for trait in OCEAN_TRAITS:
    psychometric[trait] = pd.to_numeric(
        psychometric[trait],
        errors="coerce"
    )

psychometric = (
    psychometric
    .dropna(subset=OCEAN_TRAITS)
    .drop_duplicates(subset=["user"])
)

## Calculating the OCEAN bands using training users

In [ ]:
training_users = set(
    pd.concat(
        [
            logon_features[
                logon_features["day"] < TRAIN_END_DAY
            ]["user"],

            device_features[
                device_features["day"] < TRAIN_END_DAY
            ]["user"],

            file_features[
                file_features["day"] < TRAIN_END_DAY
            ]["user"]
        ],
        ignore_index=True
    ).unique()
)

psychometric_reference = psychometric[
    psychometric["user"].isin(training_users)
].copy()

if psychometric_reference.empty:
    raise ValueError(
        "No psychometric users matched the activity datasets."
    )

low_cutoffs = psychometric_reference[
    OCEAN_TRAITS
].quantile(OCEAN_LOW_PERCENTILE)

high_cutoffs = psychometric_reference[
    OCEAN_TRAITS
].quantile(OCEAN_HIGH_PERCENTILE)

final_results = test_ensemble.merge(
    psychometric,
    on="user",
    how="left"
)

print(
    "Psychometric match rate:",
    round(
        final_results["openness"].notna().mean() * 100,
        2
    ),
    "%"
)

Psychometric match rate: 100.0 %


## assigning the psychometric categories

In [ ]:
def trait_band(value, trait):
    if pd.isna(value):
        return "Unavailable"

    if value <= low_cutoffs[trait]:
        return "Low"

    if value >= high_cutoffs[trait]:
        return "High"

    return "Typical"


for trait in OCEAN_TRAITS:
    final_results[
        f"{trait}_band"
    ] = final_results[trait].apply(
        lambda value, current_trait=trait:
        trait_band(value, current_trait)
    )


trait_names = {
    "openness": "Openness",
    "conscientiousness": "Conscientiousness",
    "extraversion": "Extraversion",
    "agreeableness": "Agreeableness",
    "neuroticism": "Neuroticism"
}


def ocean_context(row):
    if row["ensemble_flag"] == 0:
        return "Not Applied to Normal Record"

    if pd.isna(row["openness"]):
        return "Psychometric Data Unavailable"

    labels = []

    for trait in OCEAN_TRAITS:
        band = row[f"{trait}_band"]

        if band == "Low" or band == "High":
            labels.append(
                band + " " + trait_names[trait] + " Profile"
            )

    if not labels:
        return "No Extreme OCEAN Trait"

    return " | ".join(labels)


final_results["ocean_context"] = (
    final_results.apply(
        ocean_context,
        axis=1
    )
)

## Creating the final combined category

In [ ]:
def final_category(row):
    if row["ensemble_flag"] == 0:
        return "Normal"

    return (
        row["technical_category"]
        + " | "
        + row["ocean_context"]
    )


final_results["final_category"] = (
    final_results.apply(
        final_category,
        axis=1
    )
)

## Arranging and Saving the results

In [ ]:
output_columns = [
    "user",
    "day",

    "logon_score",
    "logon_percentile",
    "logon_flag",

    "device_score",
    "device_percentile",
    "device_flag",

    "file_score",
    "file_percentile",
    "file_flag",

    "ensemble_score",
    "ensemble_flag",
    "flagged_streams",
    "technical_category",

    "openness",
    "openness_band",

    "conscientiousness",
    "conscientiousness_band",

    "extraversion",
    "extraversion_band",

    "agreeableness",
    "agreeableness_band",

    "neuroticism",
    "neuroticism_band",

    "ocean_context",
    "final_category"
]

final_output = final_results[
    output_columns
].sort_values(
    "ensemble_score",
    ascending=False
).reset_index(drop=True)

RESULT_PATH = os.path.join(
    OUTPUT_FOLDER,
    "ensemble_anomaly_results.csv"
)

THRESHOLD_PATH = os.path.join(
    OUTPUT_FOLDER,
    "model_thresholds.csv"
)

final_output.to_csv(
    RESULT_PATH,
    index=False
)

threshold_table = pd.DataFrame(
    {
        "model": [
            "logon",
            "device",
            "file",
            "ensemble"
        ],
        "threshold": [
            logon_output["threshold"],
            device_output["threshold"],
            file_output["threshold"],
            ensemble_threshold
        ]
    }
)

threshold_table.to_csv(
    THRESHOLD_PATH,
    index=False
)

torch.save(
    logon_output["model"].state_dict(),
    os.path.join(
        OUTPUT_FOLDER,
        "logon_autoencoder.pth"
    )
)

torch.save(
    device_output["model"].state_dict(),
    os.path.join(
        OUTPUT_FOLDER,
        "device_autoencoder.pth"
    )
)

torch.save(
    file_output["model"].state_dict(),
    os.path.join(
        OUTPUT_FOLDER,
        "file_autoencoder.pth"
    )
)

pd.to_pickle(
    logon_output["scaler"],
    os.path.join(
        OUTPUT_FOLDER,
        "logon_scaler.pkl"
    )
)

pd.to_pickle(
    device_output["scaler"],
    os.path.join(
        OUTPUT_FOLDER,
        "device_scaler.pkl"
    )
)

pd.to_pickle(
    file_output["scaler"],
    os.path.join(
        OUTPUT_FOLDER,
        "file_scaler.pkl"
    )
)

print("Results saved to:", RESULT_PATH)
print("Thresholds saved to:", THRESHOLD_PATH)

final_output.head(20)

Results saved to: /content/CERT_r6_2_outputs/ensemble_anomaly_results.csv
Thresholds saved to: /content/CERT_r6_2_outputs/model_thresholds.csv


,user,day,logon_score,logon_percentile,logon_flag,device_score,device_percentile,device_flag,file_score,file_percentile,...,conscientiousness,conscientiousness_band,extraversion,extraversion_band,agreeableness,agreeableness_band,neuroticism,neuroticism_band,ocean_context,final_category
0,GWM3260,2010-01-23,0.822625,1.000000,1,NaN,NaN,0,NaN,NaN,...,46,High,18,Typical,18,Typical,31,Typical,High Conscientiousness Profile,Logon-Dominant Anomaly | High Conscientiousnes...
1,DNS1758,2010-01-23,1.601989,1.000000,1,NaN,NaN,0,NaN,NaN,...,37,Typical,35,Typical,15,Typical,30,Typical,No Extreme OCEAN Trait,Logon-Dominant Anomaly | No Extreme OCEAN Trait
2,MTS0465,2010-01-24,1.039044,1.000000,1,NaN,NaN,0,NaN,NaN,...,39,Typical,44,Typical,41,Typical,33,Typical,High Openness Profile,Logon-Dominant Anomaly | High Openness Profile
3,TAM3048,2010-01-24,1.819268,1.000000,1,NaN,NaN,0,NaN,NaN,...,25,Typical,39,Typical,43,Typical,29,Typical,No Extreme OCEAN Trait,Logon-Dominant Anomaly | No Extreme OCEAN Trait
4,KWC3812,2010-01-23,0.822625,1.000000,1,NaN,NaN,0,NaN,NaN,...,16,Typical,16,Typical,43,Typical,32,Typical,No Extreme OCEAN Trait,Logon-Dominant Anomaly | No Extreme OCEAN Trait
5,TAM3048,2010-01-23,1.689617,1.000000,1,NaN,NaN,0,NaN,NaN,...,25,Typical,39,Typical,43,Typical,29,Typical,No Extreme OCEAN Trait,Logon-Dominant Anomaly | No Extreme OCEAN Trait
6,MPB2578,2010-01-23,0.822625,1.000000,1,NaN,NaN,0,NaN,NaN,...,43,Typical,34,Typical,23,Typical,31,Typical,No Extreme OCEAN Trait,Logon-Dominant Anomaly | No Extreme OCEAN Trait
7,GAB3758,2010-01-23,0.519457,0.999875,1,NaN,NaN,0,NaN,NaN,...,40,Typical,23,Typical,35,Typical,26,Typical,No Extreme OCEAN Trait,Logon-Dominant Anomaly | No Extreme OCEAN Trait
8,IAS0857,2010-01-23,0.519457,0.999875,1,NaN,NaN,0,NaN,NaN,...,45,Typical,17,Typical,38,Typical,25,Typical,No Extreme OCEAN Trait,Logon-Dominant Anomaly | No Extreme OCEAN Trait
9,BMN0178,2010-01-23,0.519457,0.999875,1,NaN,NaN,0,NaN,NaN,...,42,Typical,20,Typical,14,Typical,36,Typical,No Extreme OCEAN Trait,Logon-Dominant Anomaly | No Extreme OCEAN Trait
